# Reproducing: Generative Adversarial Nets (Goodfellow et al., 2014)
arXiv:1406.2661 — generated by ArXivist

In [ ]:
!pip install torch torchvision -q

In [ ]:
import torch, torch.nn as nn
print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Model definitions
Minimax objective: $\min_G \max_D V(D,G) = \mathbb{E}_{x\sim p_{data}}[\log D(x)] + \mathbb{E}_{z\sim p_z}[\log(1-D(G(z)))]$

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, hidden=240, out_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim), nn.Sigmoid())
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, in_dim=784, hidden=1200):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(hidden, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

G = Generator().to(device)
D = Discriminator().to(device)
print('Models built. G params:', sum(p.numel() for p in G.parameters()))

## Mini training loop (synthetic data — no download needed to verify setup)

In [ ]:
import torch.optim as optim
bce = nn.BCELoss()
opt_D = optim.SGD(D.parameters(), lr=0.001, momentum=0.5)
opt_G = optim.SGD(G.parameters(), lr=0.001, momentum=0.5)

for step in range(20):
    real = torch.rand(32, 784).to(device)  # synthetic stand-in for MNIST batch
    z = torch.randn(32, 100).to(device)
    fake = G(z).detach()
    loss_d = bce(D(real), torch.ones(32,1).to(device)) + bce(D(fake), torch.zeros(32,1).to(device))
    opt_D.zero_grad(); loss_d.backward(); opt_D.step()
    z = torch.randn(32, 100).to(device)
    loss_g = bce(D(G(z)), torch.ones(32,1).to(device))
    opt_G.zero_grad(); loss_g.backward(); opt_G.step()
    if step % 5 == 0: print(f'step {step}: loss_d={loss_d.item():.4f} loss_g={loss_g.item():.4f}')
print('Mini loop complete — setup verified.')

## Full training on real MNIST
Uncomment and run for the full reproduction.

In [ ]:
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader
# transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
# train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
# loader = DataLoader(train_data, batch_size=100, shuffle=True)
# for epoch in range(50):
#     for real, _ in loader:
#         real = real.to(device)
#         z = torch.randn(real.size(0), 100).to(device)
#         fake = G(z).detach()
#         loss_d = bce(D(real), torch.ones(real.size(0),1).to(device)) + bce(D(fake), torch.zeros(real.size(0),1).to(device))
#         opt_D.zero_grad(); loss_d.backward(); opt_D.step()
#         z = torch.randn(real.size(0), 100).to(device)
#         loss_g = bce(D(G(z)), torch.ones(real.size(0),1).to(device))
#         opt_G.zero_grad(); loss_g.backward(); opt_G.step()
#     print(f'epoch {epoch}: loss_d={loss_d.item():.4f} loss_g={loss_g.item():.4f}')

## Paper's reported results (for comparison)
| Metric | Dataset | Paper value |
|---|---|---|
| Parzen log-likelihood | MNIST | 225 ± 2 |
| Parzen log-likelihood | TFD | 2057 ± 26 |

After running the full training cell above and computing your own Parzen window log-likelihood, bring your numbers back to the ArXivist chat to trigger Stage 6.